# Kumo Engine task examples with TPC-H on Databricks

Explore binary classification, regression, and link prediction with Kumo Engine on TPC-H data. 

This walkthrough reads the built-in `samples.tpch` dataset, materializes a local graph snapshot, and retains model bundles in a Unity Catalog Volume. No external credentials are required.

Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use


## Prepare your environment

**Runtime choices:** the default path is Serverless GPU; select a Python 3.12 AI Runtime GPU environment. You could also choose the classic all-purpose GPU cluster with dedicated access mode on Databricks Runtime 17.3 LTS ML (Python 3.12.3). 

**Compute and storage:** choose a CUDA-capable GPU and sufficient CPU RAM/GPU memory for the graph and model. Reserve at least four times (recommended is six times) the materialized graph-index size in free local scratch, plus room for retained runs. Use `/local_disk0/tmp/kumoai-local` for ephemeral scratch on classic dedicated compute, or `/tmp/kumoai-local` path on Serverless GPU; `/local_disk0` is not writable on serverless or standard access mode. `/tmp` is temporary and should not hold durable or sensitive exports. Use a pre-created Unity Catalog Volume under `/Volumes/<catalog>/<schema>/<volume>` for durable model bundles.

**Permissions and access:** attach/use the chosen GPU compute; install `kumo-engine[databricks]` from PyPI; use the SQL warehouse (`CAN USE`); read `samples.tpch` (`USE CATALOG`, `USE SCHEMA`, `SELECT`); create/replace views in the destination catalog/schema (`USE CATALOG`, `USE SCHEMA`, `CREATE TABLE`, plus `CREATE SCHEMA` on the catalog if this notebook creates that schema); and read/write the configured Volume (`READ VOLUME`, `WRITE VOLUME`). Enabling prediction output also requires permission to create or overwrite its dedicated table. Existing views must be owned or manageable by the running identity.

See the [Databricks Runtime 17.3 LTS ML release notes](https://docs.databricks.com/aws/en/release-notes/runtime/17.3lts-ml) and [Databricks local-file guidance](https://docs.databricks.com/aws/en/files/write-data) for current platform details.

The TPC-H dataset contains `customer`, `orders`, `lineitem`, and `part` with the columns used below. The notebook materializes local graph data and adds a synthetic primary key for `lineitem`.

## 1. Enter your configuration

Set the destination Unity Catalog catalog and schema for the views and output. The sample source remains `samples.tpch`. Kumo uses the configured local scratch directory to materialize graph and training data. A Unity Catalog Volume retains model bundles. Set the prediction date within the data's date range (1992-01-01 to 1998-08-02).


In [ ]:
from datetime import datetime
from pathlib import Path

# ── TPC-H source (Databricks sample data) ──
TPCH_CATALOG = "samples"
TPCH_SCHEMA = "tpch"

# ── Databricks connector ──
CONNECTOR_CATALOG = "<connector_catalog>"
CONNECTOR_SCHEMA = "<connector_schema"
SQL_WAREHOUSE_ID = "<sql_warehouse_id>"               # Starter Warehouse

# MODEL_NAME = "TPCH_CUSTOMER_ORDER_MODEL"
PREDICTION_TIME = datetime(1998, 1, 1)              # Orders range 1992-01-01 to 1998-08-02.
PREDICTION_OUTPUT_TABLE = "TPCH_ORDER_PREDICTIONS"

# Durable prediction output.
WRITE_PREDICTIONS = True


## 2. Install Kumo Engine from PyPI

Run the installation cell once, then restart Python. After the restart, rerun configuration and continue at imports.


In [ ]:
%pip install kumo-engine[databricks]

dbutils.library.restartPython()


<a id="runtime"></a>
## Import and configure scratch

After each kernel restart, run configuration and verify that the runtime imports and a CUDA operation succeed. They do not depend on variables created during installation. 

In [ ]:
from importlib.metadata import version
import shutil
import tempfile

from IPython.display import display
import torch
import torch_geometric

import kumo.engine as kumo
from kumo.engine import PredictionOutputConfig
from kumoapi.graph import Edge
from kumoapi.jobs import PredictionArtifactType, WriteMode
from kumoapi.task import TaskType
from kumo_databricks.output import SEPARATOR as _SEP


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is available to this notebook.")

values = torch.tensor([1.0, 2.0], device="cuda")
groups = torch.tensor([0, 0], device="cuda")
total = torch.zeros(1, device="cuda").scatter_add_(0, groups, values)
assert total.item() == 3.0

print("Kumo:", kumo.__version__)
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("PyG:", torch_geometric.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Configure scratch before constructing any Kumo table, graph, or trainer.

In [ ]:
SCRATCH_DIR = Path("/tmp/kumoai-local")
ARTIFACT_ROOT = Path("/Volumes/<catalog>/<schema>/<volume>/runs")
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=SCRATCH_DIR) as probe:
    probe.write(b"scratch check")
    probe.flush()
with tempfile.NamedTemporaryFile(dir=ARTIFACT_ROOT) as probe:
    probe.write(b"volume check")
    probe.flush()
kumo.configure(scratch_dir=str(SCRATCH_DIR))
print("Free scratch space (GiB):", round(shutil.disk_usage(SCRATCH_DIR).free / 2**30, 1))
print("Retained artifact root:", ARTIFACT_ROOT)

## Connect to TPC-H data

The notebook creates Unity Catalog views of the four TPC-H sample tables in the configured schema. The `lineitem` view adds a synthetic single-column primary key (`lineitem_id`) since TPC-H uses a composite key. A `DatabricksConnector` uses the notebook's default auth chain — no separate credentials are needed.


In [ ]:
# ── Create TPC-H views in the connector schema (SF≈1 subset) ──
# SF=1 scale: ~150K customers, ~1.5M orders, ~6M line items, ~200K parts
SF1_CUST = 150000
SF1_PART = 200000

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}")

spark.sql(f"""
    CREATE OR REPLACE VIEW {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.customer AS
    SELECT * FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.customer
    WHERE c_custkey <= {SF1_CUST}
""")
print(f"View: {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.customer (SF≈1, ≤{SF1_CUST})")

spark.sql(f"""
    CREATE OR REPLACE VIEW {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.orders AS
    SELECT * FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.orders
    WHERE o_custkey <= {SF1_CUST}
""")
print(f"View: {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.orders (SF≈1)")

spark.sql(f"""
    CREATE OR REPLACE VIEW {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.part AS
    SELECT * FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.part
    WHERE p_partkey <= {SF1_PART}
""")
print(f"View: {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.part (SF≈1, ≤{SF1_PART})")

# Lineitem: synthetic PK + filter to SF≈1 customers and parts
spark.sql(f"""
    CREATE OR REPLACE VIEW {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.lineitem AS
    SELECT *, CAST(l_orderkey * 8 + l_linenumber AS BIGINT) AS lineitem_id
    FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.lineitem
    WHERE l_orderkey IN (
        SELECT o_orderkey FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.orders
        WHERE o_custkey <= {SF1_CUST}
    )
    AND l_partkey <= {SF1_PART}
""")
print(f"View: {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.lineitem (SF≈1, with lineitem_id)")

# ── Create Kumo Databricks connector (uses notebook's default auth) ──
connector = kumo.DatabricksConnector.from_notebook(
    catalog=CONNECTOR_CATALOG,
    schema_name=CONNECTOR_SCHEMA,
    warehouse_id=SQL_WAREHOUSE_ID,
)
print("Tables:", connector.table_names())


## Describe the four tables

A primary key identifies a row. A physical type describes its stored representation, such as a date; a semantic type describes its modeling role, such as a category or numerical value. 

The customer and part definitions select the columns used in this example. 

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector["customer"],
        ["c_custkey", "c_nationkey", "c_mktsegment", "c_acctbal"],
        primary_key="c_custkey",
    )
)

part = (
    kumo.Table.from_source_table(
        connector["part"],
        ["p_partkey", "p_name", "p_brand", "p_type", "p_size",
         "p_container", "p_retailprice"],
        primary_key="p_partkey",
    )
)

Event timestamps let Kumo separate information available before a prediction boundary from future outcomes. Using future information as an input causes temporal leakage.

This sample uses `o_orderdate` for orders and `l_shipdate` for line items. The ship date represents when the item became observable in the fulfillment process and is a reasonable availability timestamp for this dataset.

In [ ]:
orders = (
    kumo.Table.from_source_table(
        connector["orders"],
        ["o_orderkey", "o_custkey", "o_orderdate", "o_orderstatus",
         "o_totalprice", "o_orderpriority", "o_shippriority"],
        primary_key="o_orderkey",
        time_column="o_orderdate",
    )
)

In [ ]:
lineitem = (
    kumo.Table.from_source_table(
        connector["lineitem"],
        ["lineitem_id", "l_orderkey", "l_partkey", "l_shipdate",
         "l_quantity", "l_extendedprice", "l_discount", "l_tax",
         "l_returnflag", "l_linestatus", "l_shipmode"],
        primary_key="lineitem_id",
        time_column="l_shipdate",
    )
)

## Connect the tables and materialize a snapshot

A **graph** describes the tables and their relationships. Each arrow below goes from a foreign-key column to the destination table's primary key:

```text
lineitem.l_partkey   -> part.p_partkey
lineitem.l_orderkey  -> orders.o_orderkey
orders.o_custkey     -> customer.c_custkey
```

Validation checks the graph definition; it is not a blanket guarantee of source-data quality. A **snapshot** materializes source data into the local graph index used for training and prediction. Snapshot creation reads the tables and consumes local disk. Reuse requires an unchanged, valid retained snapshot; loading a model does not restore a lost index.

In [ ]:
graph = kumo.Graph(
    tables=[customer, part, orders, lineitem],
    edges=[
        Edge(src_table="orders", fkey="o_custkey", dst_table="customer"),
        Edge(src_table="lineitem", fkey="l_orderkey", dst_table="orders"),
        Edge(src_table="lineitem", fkey="l_partkey", dst_table="part"),
    ],
)

graph.validate()
print("graph id:", graph.id)

In [ ]:
snapshot = graph.snapshot()
print("Snapshot:", snapshot.id)
print("Local index:", snapshot.index_path)

## Define the prediction question

Predictive Query Language (PQL) is Kumo Engine's declarative syntax for defining a predictive modeling task: it specifies the target, the entity to score, and the data context. This default query predicts how many orders a customer active in the past year will place in the next 90 days.

In [ ]:
# Regression: predict how many orders a customer active in the past year will place in the next 90 days.
pquery = kumo.PredictiveQuery(
    graph=graph,
    query=(
        "PREDICT COUNT(orders.*, 0, 90, days) "
        "FOR EACH customer.c_custkey "
        "WHERE COUNT(orders.*, -365, 0, days) > 0"
    ),
)
pquery.validate()
print("Task type:", pquery.get_task_type())

### Optional task alternatives

The next two code cells are commented out. To try one, skip or comment out the binary query cell above, uncomment exactly one alternative, and run it before training. Both define `pquery` and use the existing `graph`, `snapshot`, and downstream variables. Train a new model for the selected task. The link example ranks parts associated with orders, not customer-level recommendations.

In [ ]:
# Binary classification: predict whether a customer active in the past year will order again in the next 90 days.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query="""
#         PREDICT COUNT(orders.*, 0, 90, days) > 0
#         FOR EACH customer.c_custkey
#         WHERE COUNT(orders.*, -365, 0, days) > 0
#     """,
# )

# pquery.validate()
# print("Task type:", pquery.get_task_type())

In [ ]:
# Link prediction: rank up to five parts that may ship in the next 90 days for an order with recent line items.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT LIST_DISTINCT(lineitem.l_partkey, 0, 90, days) "
#         "RANK TOP 5 FOR EACH orders.o_orderkey "
#         "WHERE COUNT(lineitem.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

### Generate a Training Table

After your query is validated, generate a training table to use for model fitting.

In [ ]:
train_table = pquery.generate_training_table(snapshot)
print("Training table:", train_table.train_table_path)


## Create a Trainer

The suggested model plan controls architecture, sampling, and optimization. The model plan is fully customizable: inspect and adjust its settings for your task, then create `trainer = kumo.Trainer(model_plan)` before fitting.

In [ ]:
model_plan = pquery.suggest_model_plan(snapshot)
# Create a Trainer with your model plan.
trainer = kumo.Trainer(model_plan)
print(model_plan)


### Train a Model

After creating a Trainer from the model plan, call `fit()` with the training table and compatible graph snapshot.


In [ ]:
# Serverless GPU workaround for Optuna child process IPC authentication errors.
# import multiprocessing
# try:
#     multiprocessing.set_start_method("spawn", force=True)
# except RuntimeError:
#     pass

training_result = trainer.fit(train_table, snapshot=snapshot)


### Inspect the selected result

Review the reported metrics and their stage names. Available metrics vary by task and build. The retained model bundle preserves model state; it does not archive the complete experiment history.


In [ ]:
print("Available metrics:", training_result.metrics)
print("Reusable local bundle:", training_result.bundle_dir)
for name, path in training_result.artifact_paths.items():
    print(name, path)


In [ ]:
date_range = spark.sql(f"""
    SELECT MIN(o_orderdate) AS first_order, MAX(o_orderdate) AS last_order
    FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.orders
""")
display(date_range.toPandas())
print("Configured prediction boundary:", PREDICTION_TIME)

### Generate the scoring population

For these temporal queries, use a timezone-naive timestamp interpreted as UTC and reuse the fitted trainer's saved training-table generation plan. Binary classification and regression score customers with an order in the past year; the link example scores orders with recent line items. This timestamp example does not apply to static queries.

In [ ]:
prediction_table = pquery.generate_prediction_table(
    snapshot,
    prediction_time=PREDICTION_TIME,
    plan=trainer.training_table_generation_plan,
)

<a id="prediction"></a>
## Generate Batch Predictions

After training a model, generate batch predictions and write them to a Unity Catalog table. `WRITE_PREDICTIONS = True` enables that write by default; set it to `False` for local-only results.

Check the TPC-H order-date range against `PREDICTION_TIME` before scoring. A chosen date alone does not guarantee the following 90 days of observed outcomes for historical evaluation.

In [ ]:
# Specific threshold or candidate count are required for binary classification and link prediction tasks respectively. 
# prediction_options = {}
# task_type = pquery.get_task_type()
# if task_type == TaskType.BINARY_CLASSIFICATION:
#     prediction_options["binary_classification_threshold"] = 0.5
# elif task_type.is_link_pred:
#     prediction_options["num_classes_to_return"] = 10

output_name = f"{CONNECTOR_SCHEMA}{_SEP}{PREDICTION_OUTPUT_TABLE}"

output_config = None
if WRITE_PREDICTIONS:
    output_config = PredictionOutputConfig(
        connector=connector,
        artifact_type=PredictionArtifactType.PREDICTIONS,
        table_name=output_name,
        write_mode=WriteMode.OVERWRITE,
    )
result = trainer.predict(
    prediction_table,
    snapshot=snapshot,
    output_config=output_config,
)

print("Output rows:", result.num_rows)
if WRITE_PREDICTIONS:
    print("Output table:", f"{CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.{PREDICTION_OUTPUT_TABLE}")
else:
    print("Local prediction files:", result.files)